In [1]:
!pip freeze -> requirements.txt
from __future__ import annotations

import csv
import json
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
"""
Step 4B: Scrape AbeBooks listings by ISBN.

Input:
  data/raw/isbn_seed_1000.csv   (column: isbn)

Output:
  data/raw/abebooks_raw.jsonl   (one JSON object per listing)
  data/raw/abebooks_raw.csv     (all listings in a single CSV)

Notes:
- Scraping can break if AbeBooks changes HTML.
- Be polite: sleep between requests, set User-Agent, handle failures.
"""



ABEBOKS_SEARCH_URL = "https://www.abebooks.com/servlet/SearchResults"


# ---------------------------
# Helpers
# ---------------------------

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def safe_get(url: str, params: dict, timeout: int = 25, retries: int = 3, sleep_backoff: float = 1.0) -> Optional[str]:
    """GET HTML with retries and simple backoff. Returns HTML text or None."""
    headers = {
        "User-Agent": "DSCI510-BookPriceProject/1.0 (educational use)",
        "Accept-Language": "en-US,en;q=0.9",
    }

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            if r.status_code == 200 and r.text:
                return r.text
            # 404/403/429 etc.
            print(f"[WARN] HTTP {r.status_code} for params={params}")
        except requests.RequestException as e:
            print(f"[WARN] Request error attempt {attempt}/{retries}: {e}")

        time.sleep(sleep_backoff * attempt)

    return None


def parse_price(text: str) -> Tuple[Optional[float], Optional[str]]:
    """
    Parse something like '$12.34' or 'US$ 12.34' into (12.34, 'USD' guess).
    AbeBooks can show different currency formats; we'll do best-effort.
    """
    if not text:
        return None, None

    t = text.strip()

    # currency hint
    currency = None
    if "US$" in t or "$" in t:
        currency = "USD"
    elif "£" in t:
        currency = "GBP"
    elif "€" in t:
        currency = "EUR"

    # extract first number like 12.34
    m = re.search(r"(\d+(?:\.\d{1,2})?)", t.replace(",", ""))
    if not m:
        return None, currency

    try:
        return float(m.group(1)), currency
    except ValueError:
        return None, currency


def normalize_condition(text: Optional[str]) -> Optional[str]:
    """Standardize condition labels into a few buckets."""
    if not text:
        return None
    t = text.strip().lower()
    if "new" in t:
        return "new"
    if "like new" in t:
        return "like_new"
    if "very good" in t:
        return "used_very_good"
    if "good" in t:
        return "used_good"
    if "acceptable" in t:
        return "used_acceptable"
    if "used" in t:
        return "used"
    return t.replace(" ", "_")


@dataclass
class Listing:
    source: str
    isbn: str
    title: Optional[str]
    author: Optional[str]
    listing_price: Optional[float]
    shipping_price: Optional[float]
    total_price: Optional[float]
    currency: Optional[str]
    condition: Optional[str]
    rating: Optional[float]
    review_count: Optional[int]
    url: Optional[str]
    scrape_time_utc: str



In [3]:

# ---------------------------
# AbeBooks parsing
# ---------------------------

def parse_abebooks_search(html: str, isbn: str) -> List[Listing]:
    """
    Parse AbeBooks SearchResults page for a given ISBN.
    HTML structure can change; this is best-effort.

    Strategy:
    - Find listing containers.
    - Extract title, author, price, shipping (if present), condition, url.
    """
    soup = BeautifulSoup(html, "lxml")

    scrape_time = utc_now_iso()

    listings: List[Listing] = []

    # AbeBooks often uses "result" containers; selectors may vary.
    # We'll try a few common patterns.
    result_cards = soup.select("[data-cy='listing-item']")  # sometimes present
    if not result_cards:
        result_cards = soup.select(".result, .result-item, .cf.result, .srp-item")  # fallback guesses

    # If still nothing, don't crash — just return empty.
    if not result_cards:
        return listings

    for card in result_cards[:25]:  # cap per ISBN so you don't explode the dataset
        # Title + URL
        title = None
        url = None

        a = card.select_one("a")
        if a and a.get_text(strip=True):
            title = a.get_text(strip=True)
            href = a.get("href")
            if href:
                url = href if href.startswith("http") else f"https://www.abebooks.com{href}"

        # Author (best effort)
        author = None
        author_el = card.select_one("[data-cy='author'], .author, .author-name")
        if author_el:
            author = author_el.get_text(" ", strip=True)

        # Condition (best effort)
        condition = None
        cond_el = card.select_one("[data-cy='condition'], .condition, .item-condition")
        if cond_el:
            condition = normalize_condition(cond_el.get_text(" ", strip=True))

        # Price (best effort)
        listing_price = None
        currency = None
        price_el = card.select_one("[data-cy='price'], .price, .item-price")
        if price_el:
            listing_price, currency = parse_price(price_el.get_text(" ", strip=True))

        # Shipping (optional, often hard to get reliably)
        shipping_price = None
        ship_el = card.select_one("[data-cy='shipping'], .shipping, .item-shipping")
        if ship_el:
            shipping_price, ship_currency = parse_price(ship_el.get_text(" ", strip=True))
            # if currency missing, inherit
            if not currency and ship_currency:
                currency = ship_currency

        total_price = None
        if listing_price is not None and shipping_price is not None:
            total_price = listing_price + shipping_price
        elif listing_price is not None:
            total_price = listing_price

        listings.append(
            Listing(
                source="abebooks",
                isbn=isbn,
                title=title,
                author=author,
                listing_price=listing_price,
                shipping_price=shipping_price,
                total_price=total_price,
                currency=currency,
                condition=condition,
                rating=None,         # AbeBooks often doesn't expose rating consistently
                review_count=None,   # same
                url=url,
                scrape_time_utc=scrape_time,
            )
        )

    return listings



In [5]:

# ---------------------------
# Pipeline
# ---------------------------

def scrape_abebooks_for_isbn(isbn: str) -> List[Listing]:
    params = {"isbn": isbn}
    html = safe_get(ABEBOKS_SEARCH_URL, params=params)
    if not html:
        return []
    return parse_abebooks_search(html, isbn)


def main() -> None:
    isbn_path = "data/raw/isbn_seed_1000.csv"
    out_jsonl = "data/raw/abebooks_raw.jsonl"
    out_csv = "data/raw/abebooks_raw.csv"

    df = pd.read_csv(isbn_path)
    isbns = df["isbn"].astype(str).dropna().unique().tolist()

    all_rows: List[Dict] = []

    # Write JSONL as we go (so you don’t lose progress)
    with open(out_jsonl, "w", encoding="utf-8") as f_jsonl:
        for i, isbn in enumerate(isbns, start=1):
            print(f"[INFO] ({i}/{len(isbns)}) Scraping AbeBooks for ISBN: {isbn}")

            listings = scrape_abebooks_for_isbn(isbn)

            for listing in listings:
                row = asdict(listing)
                f_jsonl.write(json.dumps(row, ensure_ascii=False) + "\n")
                all_rows.append(row)

            # polite delay to reduce blocking risk
            time.sleep(1.0)

    # Save a CSV for easy inspection
    if all_rows:
        pd.DataFrame(all_rows).to_csv(out_csv, index=False)
        print(f"[DONE] Saved {len(all_rows)} listings to:\n  {out_jsonl}\n  {out_csv}")
    else:
        print("[DONE] No listings captured (possible HTML change / blocking / network issues).")


if __name__ == "__main__":
    main()


[INFO] (1/1000) Scraping AbeBooks for ISBN: 9780486227818
[INFO] (2/1000) Scraping AbeBooks for ISBN: 9780099537878
[INFO] (3/1000) Scraping AbeBooks for ISBN: 9781411435049
[INFO] (4/1000) Scraping AbeBooks for ISBN: 9781169232495
[INFO] (5/1000) Scraping AbeBooks for ISBN: 9781072826040
[INFO] (6/1000) Scraping AbeBooks for ISBN: 9780606307925
[INFO] (7/1000) Scraping AbeBooks for ISBN: 9780448448862
[INFO] (8/1000) Scraping AbeBooks for ISBN: 9798352499320
[INFO] (9/1000) Scraping AbeBooks for ISBN: 9780394747095
[INFO] (10/1000) Scraping AbeBooks for ISBN: 9798591487577
[INFO] (11/1000) Scraping AbeBooks for ISBN: 9781789430905
[INFO] (12/1000) Scraping AbeBooks for ISBN: 9780140364545
[INFO] (13/1000) Scraping AbeBooks for ISBN: 9780140059052
[INFO] (14/1000) Scraping AbeBooks for ISBN: 9780425195208
[INFO] (15/1000) Scraping AbeBooks for ISBN: 9781015437890
[INFO] (16/1000) Scraping AbeBooks for ISBN: 9781096133315
[INFO] (17/1000) Scraping AbeBooks for ISBN: 9788310109392
[INFO]